In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json_tricks

In [ ]:
inputs = json_tricks.load('inputs/inputs.json')

# Task

In the input file, you are given tables of statistical weights $M(\xi=x, \eta=y)$ of random variables, together with their possible values.

Your task is to:

0) Find the distribution $p(\xi=x, \eta=y)$
1) Find the distribution $p(\xi = x)$
2) Find the distribution $p(\eta = y)$
3) Check if $\xi$ and $\eta$ are independent (the definition of independence is required)
4) Find the conditional distribution $p(\xi=x|\eta=y)$
5) Find the conditional distribution $p(\eta=y|\xi=x)$ (this one can be done using Bayes' theorem)
6) Find $\mathbb E \xi$
7) Find $\mathbb E \eta$
8) Find $\mathbb E (\xi + \eta)$
8) Find $\mathbb E (\xi \eta)$
9) Find $\mathbb E \exp(\xi + \eta)$
10) Find $\mathbb V \xi$
11) Find $\mathbb V \eta$
11) Find $\mathbb V (\eta + \xi)$
12) Find $\sigma_{\xi}$
13) Find $\sigma_{\eta}$
12) Find $\mathrm{cov} (\xi, \eta)$

If you need to return a NumPy array, align it with the input $M$ array. This means 
that if `M[i, j]` is the measure of event of $\xi = x_i, \eta = y_j$, then the corresponding
`p[i, j]` is the probability of the corresponding event, `p_x[i]` is the probability of $\xi = x_i$
and `p_x[j]` is the probability of $\eta = y_j$.

In [ ]:
def task(measure, xvals, yvals):
    p = measure / np.sum(measure)
    p_x = np.sum(p, axis=1)
    p_y = np.sum(p, axis=0)
    p_expected_indep = p_x[:, np.newaxis] * p_y[np.newaxis, :]
    independency = bool(np.allclose(p, p_expected_indep, atol=1e-9))
    p_y_giv_x = np.where(p_y[np.newaxis, :] > 0, p / p_y[np.newaxis, :], 0.0)
    p_x_giv_y = np.where(p_x[:, np.newaxis] > 0, p / p_x[:, np.newaxis], 0.0)
    Ex = float(np.sum(xvals * p_x))
    Ey = float(np.sum(yvals * p_y))
    Ex_plus_y = Ex + Ey
    xy_grid = xvals[:, np.newaxis] * yvals[np.newaxis, :]
    Ex_times_y = float(np.sum(xy_grid * p))
    x_plus_y_grid = xvals[:, np.newaxis] + yvals[np.newaxis, :]
    Eexp = float(np.sum(np.exp(x_plus_y_grid) * p))
    Vx = float(np.sum(((xvals - Ex) ** 2) * p_x))
    Vy = float(np.sum(((yvals - Ey) ** 2) * p_y))
    Vx_plus_y = float(np.sum(((x_plus_y_grid - Ex_plus_y) ** 2) * p))
    sigmax = float(np.sqrt(Vx))
    sigmay = float(np.sqrt(Vy))
    covxy = float(Ex_times_y - Ex * Ey)
    linear_independency = bool(np.isclose(covxy, 0.0, atol=1e-9))

    return {
        'p': p,
        'p_x': p_x,
        'p_y': p_y,
        'independency': independency,
        'p_x_giv_y': p_x_giv_y,
        'p_y_giv_x': p_y_giv_x,
        'Ex': Ex,
        'Ey': Ey,
        'Ex_plus_y': Ex_plus_y,
        'Ex_times_y': Ex_times_y,
        'Eexp': Eexp,
        'Vx': Vx,
        'Vy': Vy,
        'Vx_plus_y': Vx_plus_y,
        'sigmax': sigmax,
        'sigmay': sigmay,
        'covxy': covxy,
        'linear_independency': linear_independency
    }

In [ ]:
answer = []
for input in inputs:
    answer.append(task(**input))

json_tricks.dump(answer, '.answer.json')

# Visualization: Two Dice

The first task corresponds to the simultaneous rolling of two dice, as you do in Monopoly or Catan.

Let us visualize the distribution of $\xi + \eta$, its mean, and its standard deviation.

Does this make sense? How can we utilize this information when playing Catan or Monopoly?

In [ ]:
two_dice = answer[0]

print(two_dice['p'])

x_plus_y_p = {}
for x_index in range(two_dice['p'].shape[0]):
    for y_index in range(two_dice['p'].shape[1]):
        if x_index + 1 + y_index + 1 not in x_plus_y_p:
            x_plus_y_p[x_index + 1 + y_index + 1] = 0
        x_plus_y_p[x_index + 1 + y_index + 1] += two_dice['p'][x_index, y_index]

# print(x_plus_y_p)
# x_plus_y_p = dict(sorted(x_plus_y_p))
# print(x_plus_y_p)
plt.plot(list(x_plus_y_p.keys()), list(x_plus_y_p.values()), 'ro')
plt.axvline(x=two_dice['Ex_plus_y'], color='r', linestyle='--', linewidth=2)
plt.axvline(x=two_dice['Ex_plus_y'] - np.sqrt(two_dice['Vx_plus_y']), color='b', linestyle='--', linewidth=2)
plt.axvline(x=two_dice['Ex_plus_y'] + np.sqrt(two_dice['Vx_plus_y']), color='b', linestyle='--', linewidth=2)
plt.legend(['distribution', 'expectance', 'deviation'])

# Interesting fact

Note that the second test case is about nonlinearly dependent random variables.

Let us check:
- covariance
- independence
- linear independence

In [ ]:
two_dice = answer[1]

print(two_dice['covxy'])
print(two_dice['linear_independency'])
print(two_dice['independency'])

Indeed, this distribution is dependent, but not linearly dependent.